# Version 2

In [2]:
import pandas as pd
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import textstat

# Download required NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')

# Function to compute scores
def compute_scores(ground_truth, summary):
    # Tokenize the inputs
    ground_truth_tokens = ground_truth.split()
    summary_tokens = summary.split()

    # Compute BLEU score
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([ground_truth_tokens], summary_tokens, smoothing_function=smoothing_function)
    
    # Compute METEOR score
    meteor = meteor_score([ground_truth_tokens], summary_tokens)
    
    # Compute ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ground_truth, summary)
    rouge1 = scores['rouge1'].fmeasure
    rouge2 = scores['rouge2'].fmeasure
    rougeL = scores['rougeL'].fmeasure
    
    # Compute Flesch Reading Ease score
    flesch_reading_ease = textstat.flesch_reading_ease(summary)
    
    return bleu_score, meteor, rouge1, rouge2, rougeL, flesch_reading_ease

def main(input_csv, ground_truth_text, output_csv):
    # Read input CSV
    df = pd.read_csv(input_csv)
    
    # Initialize lists to store scores for BART summaries
    bart_bleu_scores = []
    bart_meteor_scores = []
    bart_rouge1_scores = []
    bart_rouge2_scores = []
    bart_rougeL_scores = []
    bart_flesch_reading_ease_scores = []

    # Initialize lists to store scores for BERT summaries
    bert_bleu_scores = []
    bert_meteor_scores = []
    bert_rouge1_scores = []
    bert_rouge2_scores = []
    bert_rougeL_scores = []
    bert_flesch_reading_ease_scores = []
    
    # Compute scores for BART summaries
    for summary in df['Summary (BART) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bart_bleu_scores.append(bleu)
        bart_meteor_scores.append(meteor)
        bart_rouge1_scores.append(rouge1)
        bart_rouge2_scores.append(rouge2)
        bart_rougeL_scores.append(rougeL)
        bart_flesch_reading_ease_scores.append(flesch_reading_ease)

    # Compute scores for BERT summaries
    for summary in df['Summary (BERT) Reduced']:
        bleu, meteor, rouge1, rouge2, rougeL, flesch_reading_ease = compute_scores(ground_truth_text, summary)
        bert_bleu_scores.append(bleu)
        bert_meteor_scores.append(meteor)
        bert_rouge1_scores.append(rouge1)
        bert_rouge2_scores.append(rouge2)
        bert_rougeL_scores.append(rougeL)
        bert_flesch_reading_ease_scores.append(flesch_reading_ease)
    
    # Add scores to DataFrame
    df['BLEU (BART)'] = bart_bleu_scores
    df['METEOR (BART)'] = bart_meteor_scores
    df['ROUGE-1 (BART)'] = bart_rouge1_scores
    df['ROUGE-2 (BART)'] = bart_rouge2_scores
    df['ROUGE-L (BART)'] = bart_rougeL_scores
    df['Flesch Reading Ease (BART)'] = bart_flesch_reading_ease_scores

    df['BLEU (BERT)'] = bert_bleu_scores
    df['METEOR (BERT)'] = bert_meteor_scores
    df['ROUGE-1 (BERT)'] = bert_rouge1_scores
    df['ROUGE-2 (BERT)'] = bert_rouge2_scores
    df['ROUGE-L (BERT)'] = bert_rougeL_scores
    df['Flesch Reading Ease (BERT)'] = bert_flesch_reading_ease_scores
    
    # Save DataFrame to new CSV
    df.to_csv(output_csv, index=False)
    
if __name__ == "__main__":
    input_csv = 'CSV/Refining/UnicodeR1Janus.csv'  
    ground_truth_text = """The agent-based model used in this study aims to replicate the adoption behaviors of milk consumption by the British public from 1974 to 2005. The model examines how milk choice decisions change as a function of individual preferences and decision factors including environmental and health values, habits, and social influences. The goal is to replicate a historical trend showing the substitution of whole milk with skimmed and semi-skimmed milk. The model employs both a threshold-based and a probability-based approach to determine how agents decide on milk choices. The simulation starts with agents who initially prefer whole milk. Key parameters set in the model include the number of agents at 1000, the habit mechanism enabled, and the use of a Watts-Strogatz small-world network with an average of 8 connections per agent to simulate social influences. The model incorporates social norms and equips agents with a memory consisting of their previous five steps. Agents develop habits if they make the same choice for two consecutive steps, and the cognitive dissonance threshold is set at 0.15 out of 1. Connected agents have a 30% chance to interact at a given step, while agents exhibit a 51% tendency to ignore social information. Social conformity and susceptibility to peer influence are set at 0.28 out of 1 and 0.2 out of 1, respectively. Initially, agents perceive the health and environmental benefits of alternative milk as 1.69 out of 3 and 1.12 out of 3, and they start with a strong initial habit of consuming whole milk (10 out of 10). Justification for choices is set at 0.61 out of 1, reflecting their ability to rationalize decisions.

At the start of the simulation, whole milk consumption is very high, mirroring historical data where whole milk was the dominant choice among the British public. In the early stage (up to about time step 10) there is a rapid decline in whole milk consumption. After time step 10, there is a slower but steady rate of decline. Meanwhile, the mean consumption of skimmed/semi-skimmed milk starts very low but increases rapidly between time steps 5 and 10, showing early adopters' influence. This increase stabilizes after time step 10, with a steady but slower rise, suggesting gradual adoption by the remaining population.

The total number of choices for whole milk shows a steady and consistent increase to finally reach 2000 choices for whole milk. This linear trend suggests that the rate of choosing whole milk remains constant. Conversely, the total number of choices for skimmed/semi-skimmed milk also increases consistently and steadily, forming a nearly perfect linear trend reaching about 25000 choices for these milk options. This reflects a growing preference over time for these milk options compared to whole milk. The total choices made by agents show a uniform and continuous decision-making process, with a consistent increase throughout the simulation, highlighting a steady adoption of skimmed milk and semi-skimmed milk among the population.

We observe a slight increase in the population’s preference for whole milk, as the mean habit factor for whole milk starts slightly above 1.00 and increases gradually up to 1.05. For skimmed/semi-skimmed milk, the mean habit factor starts slightly above 1.00, plateaus initially, and then increases consistently up to 1.6, which reveals the reinforcement in this preference over whole milk. The mean deviation between the agents' held health values and the implied values by their milk choices fluctuates across simulation runs, starting with a low value and increasing steadily, thus suggesting a growing divergence between agents' health values and their consumption habits. Similarly, the difference between the agents’ environmental values and the implied values of their milk choices deviates across simulation runs, beginning with a low value that rises sharply, indicating a significant and persistent misalignment between agents' environmental values and their product choices, stabilizing at a high level."""
    output_csv = 'CSV/Refining/ScoresR1Janus.csv'
    main(input_csv, ground_truth_text, output_csv)


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/noeflandre/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
